In [6]:
import time
import uuid
import functools
from typing import Dict, Any, List

In [7]:
# This global dictionary acts as our central log storage (in production, this goes to a database)
TRACE_STORE: Dict[str, List[Dict[str, Any]]] = {}

def trace_step(step_name: str):
    """
    A decorator that wraps any function to log its inputs, outputs, execution time, 
    and errors into a central trace history.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # 1. Create a unique ID for this specific execution span
            span_id = str(uuid.uuid4())
            start_time = time.time()
            
            # Capture the inputs passed to the function
            inputs = {"args": args, "kwargs": kwargs}
            print(f"[TRACE - START] Executing '{step_name}' (Span ID: {span_id})...")
            
            try:
                # 2. Execute the actual function
                result = func(*args, **kwargs)
                duration = time.time() - start_time
                
                # 3. Log successful execution data
                log_entry = {
                    "span_id": span_id,
                    "step_name": step_name,
                    "status": "SUCCESS",
                    "inputs": inputs,
                    "output": result,
                    "duration_seconds": round(duration, 4)
                }
                return result
                
            except Exception as e:
                duration = time.time() - start_time
                # 4. Log failure details if the step crashes
                log_entry = {
                    "span_id": span_id,
                    "step_name": step_name,
                    "status": "FAILED",
                    "inputs": inputs,
                    "error": str(e),
                    "duration_seconds": round(duration, 4)
                }
                raise e # Re-throw the exception so the application logic doesn't swallow it
                
            finally:
                # Store the log entry under the current active trace pipeline
                # (For simplicity, we append everything to a flat list for this run)
                if "current_run" not in TRACE_STORE:
                    TRACE_STORE["current_run"] = []
                TRACE_STORE["current_run"].append(log_entry)
                print(f"[TRACE - END] Finished '{step_name}' in {log_entry['duration_seconds']}s")
                
        return wrapper
    return decorator

In [8]:
# --- Simulated Pipeline Components ---

@trace_step(step_name="Format Prompt Template")
def format_prompt(user_topic: str) -> str:
    """Takes raw user input and wraps it in structured system instructions."""
    return f"System: You are an expert biologist. Explain this topic concisely.\nUser: Tell me about {user_topic}"

@trace_step(step_name="Mock LLM Call")
def mock_llm_api(prompt: str) -> str:
    """Simulates a network call to an external LLM provider like OpenAI or Anthropic."""
    time.sleep(0.4) # Simulate network lag
    if not prompt:
        raise ValueError("Prompt cannot be empty strings.")
    return "Mitochondria is the powerhouse of the cell."


In [9]:
# --- Running the Chain ---

def run_ai_feature(topic: str):
    print("--- Starting AI Execution Chain ---")
    try:
        # Step 1: Format the input data
        prompt = format_prompt(user_topic=topic)
        
        # Step 2: Pass the formatted output directly into the LLM
        response = mock_llm_api(prompt=prompt)
        
        print("\nFinal Output to User:", response)
    except Exception as err:
        print(f"\nPipeline crashed due to error: {err}")
    
    # Let's inspect the historical trace data generated by our decorators
    print("\n" + "="*50)
    print("INSPECTING THE TRACE LOGS:")
    print("="*50)
    for span in TRACE_STORE["current_run"]:
        print(f"\nStep: {span['step_name']} | Status: {span['status']} | Duration: {span['duration_seconds']}s")
        print(f"  Inputs:  {span['inputs']}")
        if "output" in span:
            print(f"  Outputs: {span['output']}")
        if "error" in span:
            print(f"  Error!!: {span['error']}")

In [10]:
# Execute the application trace
run_ai_feature(topic="Cellular Energy")

--- Starting AI Execution Chain ---
[TRACE - START] Executing 'Format Prompt Template' (Span ID: 1f4df0ba-6924-481f-9015-8455dcd905a2)...
[TRACE - END] Finished 'Format Prompt Template' in 0.0s
[TRACE - START] Executing 'Mock LLM Call' (Span ID: a5c3da9f-d6b4-4677-a123-9af06c2e747a)...
[TRACE - END] Finished 'Mock LLM Call' in 0.4001s

Final Output to User: Mitochondria is the powerhouse of the cell.

INSPECTING THE TRACE LOGS:

Step: Format Prompt Template | Status: SUCCESS | Duration: 0.0s
  Inputs:  {'args': (), 'kwargs': {'user_topic': 'Cellular Energy'}}
  Outputs: System: You are an expert biologist. Explain this topic concisely.
User: Tell me about Cellular Energy

Step: Mock LLM Call | Status: SUCCESS | Duration: 0.4001s
  Inputs:  {'args': (), 'kwargs': {'prompt': 'System: You are an expert biologist. Explain this topic concisely.\nUser: Tell me about Cellular Energy'}}
  Outputs: Mitochondria is the powerhouse of the cell.
